In [13]:
import os
import openslide as ops
import pyvips
from glob import glob
from tqdm import tqdm
import numpy as np  
import matplotlib.pyplot as plt
from pathlib import Path
import importlib
import wsi_registration
importlib.reload(wsi_registration)
from wsi_registration import (
    affine_parameters, plot_registration, register_and_save_pairs,
    register_wsi_pair, transform_points
)

In [ ]:
HnE_slide_dir = Path('../../data/HnE_n_UNStaining/C_Stained-tiff')
unstain_slide_dir = Path('../../data/HnE_n_UNStaining/Un-Stained-tiff')

HnE_by_name = {path.name: path for path in HnE_slide_dir.glob('*.tiff')}
unstain_by_name = {path.name: path for path in unstain_slide_dir.glob('*.tiff')}
common_names = sorted(HnE_by_name.keys() & unstain_by_name.keys())

HnE_slide_list = [HnE_by_name[name] for name in common_names]
unstain_slide_list = [unstain_by_name[name] for name in common_names]
print(f'Matched slide pairs: {len(common_names)}')

In [ ]:
idx = 2
registration = register_wsi_pair(
    moving_path=HnE_slide_list[idx],
    fixed_path=unstain_slide_list[idx],
    max_size=1000,
)

print(f'Pair: {common_names[idx]}')
print(f'Unstain mask sensitivity: +{registration.fixed_mask_threshold_offset}')
print(f'Mask IoU: {registration.iou_before:.4f} -> {registration.iou_after:.4f}')
print(f'ECC score: {registration.ecc_score:.4f}')
print('Transform:', affine_parameters(registration.matrix_moving_to_fixed_thumbnail))
plot_registration(registration, moving_label='H&E', fixed_label='Unstain')

In [ ]:
# H&E level-0 좌표를 Unstain level-0 좌표로 변환하는 3x3 행렬
registration.matrix_moving_to_fixed_fullres

In [ ]:
registration_output_dir = Path('../../data/HnE_n_UNStaining/registration_results')
batch_summary = register_and_save_pairs(
    moving_paths=HnE_slide_list,
    fixed_paths=unstain_slide_list,
    output_dir=registration_output_dir,
    max_size=1000,
    overwrite=False,  # 이미 완료된 결과는 건너뛰고 이어서 실행
)

completed = sum(row['status'] == 'completed' for row in batch_summary)
skipped = sum(row['status'] == 'skipped' for row in batch_summary)
errors = [row for row in batch_summary if row['status'] == 'error']
print(f'completed={completed}, skipped={skipped}, errors={len(errors)}')
errors[:5]